In [2]:
import pandas as pd
import numpy as np
import pickle
from os import listdir
from os.path import isfile, join

from utilities import *

In [3]:
folder = '../data/Apply_MI_Zheng_2023_01/'
onlyfiles = [f for f in listdir(folder) if isfile(join(folder, f))]
files = [f for f in onlyfiles if not f.startswith('._')]

In [52]:
dataSet = splitBackgroundAndDynamicData(cleanDataFolder=folder, dataFileNames=files, idColumn='SCCRIP_ID')

In [53]:
print('background: ', len(dataSet['background']), 'dynamic: ',len(dataSet['dynamic']))

background:  22 dynamic:  33


In [54]:
features = {'background': [],
            'dynamic': []
           }
for c in dataSet.keys():
    data = dataSet[c]
    for d in data:
        fname = folder + d + '.sas7bdat'
        df = read_sas(fname)
        features[c].extend(list(df.columns))   
    

In [55]:
print('background: ', len(features['background'])-len(dataSet['background']), 'dynamic: ',len(features['dynamic'])-len(dataSet['dynamic']))

background:  1570 dynamic:  1056


In [56]:
background = dataSet['background']
df_background = mergeBackgroundDataset(cleanDataFolder=folder, backgroundDatasetList=background)

/research/rgs01/home/clusterHome/mzheng09/Projects/SCCRIP/SCCRIP/Final/codes/utilities.py:120: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(lambda col: pd.to_numeric(col, errors='ignore')


In [57]:
dynamic = dataSet['dynamic']
df_dynamic = mergeDynamicDataset(cleanDataFolder=folder, dynamicDatasetList=dynamic)

/research/rgs01/home/clusterHome/mzheng09/Projects/SCCRIP/SCCRIP/Final/codes/utilities.py:152: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df = df.apply(lambda col: pd.to_numeric(col, errors='ignore')


In [58]:
df_background, df_dynamic = removeUncompleteId(df_background=df_background, df_dynamic=df_dynamic)

In [59]:
df_dynamic.drop(['Genotype_cbc'], inplace=True, axis=1)
df_background.drop(['DX_voxelotor'], inplace=True, axis=1)
df_background['Genotype_demographic'] = df_background['Genotype_demographic'].fillna(df_background['Genotype_demographic_special'])
df_background.drop(['Genotype_demographic_special'],inplace=True, axis=1)
df_background['DX_demographic'] = df_background['DX_demographic'].fillna(df_background['DX_demographic_special'])
df_background.drop(['DX_demographic_special'],inplace=True, axis=1)

df_background['Gender_demographic'] = df_background['Gender_demographic'].fillna(df_background['Gender_demographic_special'])
df_background.drop(['Gender_demographic_special'],inplace=True, axis=1)
df_background['Age_2021_demographic'] = df_background['Age_2021_demographic'].fillna(df_background['Age_2021_demographic_special'])
df_background.drop(['Age_2021_demographic_special'],inplace=True, axis=1)
df_background['Cohort_21Freeze_demographic'] = df_background['Cohort_21Freeze_demographic'].fillna(df_background['Cohort_21Freeze_demographic_special'])
df_background.drop(['Cohort_21Freeze_demographic_special'],inplace=True, axis=1)
df_background['Follow_interval_demographic'] = df_background['Follow_interval_demographic'].fillna(df_background['Follow_interval_demographic_special'])
df_background.drop(['Follow_interval_demographic_special'],inplace=True, axis=1)
df_background['Race_demographic'] = df_background['Race_demographic'].fillna(df_background['Race_demographic_special'])
df_background.drop(['Race_demographic_special'],inplace=True, axis=1)
df_background['Ethnicity_demographic'] = df_background['Ethnicity_demographic'].fillna(df_background['Ethnicity_demographic_special'])
df_background.drop(['Ethnicity_demographic_special'],inplace=True, axis=1)
df_background['Zip_code_demographic'] = df_background['Zip_code_demographic'].fillna(df_background['ZIP_demographic_special'])
df_background.drop(['ZIP_demographic_special'],inplace=True, axis=1)
df_background['DeathDt_demographic'] = df_background['DeathDt_demographic'].fillna(df_background['DeathDt_demographic_special'])
df_background.drop(['DeathDt_demographic_special'],inplace=True, axis=1)
df_background['Sex_demographic'] = df_background['Sex_demographic'].fillna(df_background['Sex_demographic_special'])
df_background.drop(['Sex_demographic_special'],inplace=True, axis=1)

In [5]:
result_dir = '../results/data/'
ensure_directory_exists(result_dir)

Directory '../results/data/' already exists.


In [61]:
df_dynamic.to_pickle(result_dir + 'dynamic.pkl')
df_background.to_pickle(result_dir + 'background.pkl')

In [62]:
len(set(df_dynamic['SCCRIP_ID']).intersection(set(df_background['SCCRIP_ID'])))

1331

In [63]:
df_background.shape

(1332, 1540)

In [64]:
df_dynamic.shape

(5447722, 945)

In [65]:
del df_background
del df_dynamic

In [52]:
############################################

In [53]:
dy_file = result_dir + 'dynamic.pkl'
ba_file = result_dir + 'background.pkl'

In [55]:
dy = pd.read_pickle(dy_file)

In [56]:
dy_ = copy.deepcopy(dy)
dy_.dropna(how='all', axis=1, inplace=True) 
mapping_dict_dy = encode_object_columns(dy_)

with open(result_dir + 'mapping_dict_dynamic.pkl', 'wb') as f:
    pickle.dump(mapping_dict_dy, f)

In [57]:
dy_.set_index('Date',inplace=True)
date_col = []
for c in dy_.columns:
    if dy_[c].dtype == 'datetime64[ns]':
        date_col.append(c)    

dy_.drop(date_col, inplace=True, axis=1)

In [58]:
age_col = []
for c in dy_:
    c_low = c.lower()
    
    if 'age' in c_low and 'initial' not in c_low:
        # print (c, dy_[c].count())        
        age_col.append(c)
age_col_copy = copy.deepcopy(age_col)
age_col.remove('Image_Quality_echo_elysis')
dy_.drop('Image_Quality_echo_elysis',axis=1, inplace=True)

a_drop = []
for i in age_col:
    if 'age_matched' in i.lower() or 'spine' in i.lower():
        a_drop.append(i)
for j in a_drop:
    age_col.remove(j)
dy_.drop(a_drop,axis=1, inplace=True)
dy_['AGE_Y_eGFR_egfr'] = dy_['AGE_Y_eGFR_egfr'].abs()
dy_['AGE_Y_HsCRP_hscrp' ] = dy_['AGE_Y_HsCRP_hscrp'].abs()
dy_['Ages'] = dy_[age_col].mean(axis=1)

dy_.drop(age_col, inplace=True, axis=1)

/lsf_tmp/272327336.tmpdir/ipykernel_693546/3671738315.py:21: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  dy_['Ages'] = dy_[age_col].mean(axis=1)


In [59]:
sex_col = []
for c in dy_.columns:
    if 'Sex' in c or 'sex' in c:
        sex_col.append(c)
dy_.drop(sex_col, inplace=True, axis=1)

In [60]:
dy_ = dy_.reset_index()
dy_.rename({'Date': 'date'}, axis=1, inplace = True)
dy_ = dy_.groupby(['SCCRIP_ID','date', 'Ages']).mean().reset_index()

dy_ = dy_.drop_duplicates()

In [61]:
dy_.to_pickle(result_dir + 'ini_proce_dynamic.pkl')

In [62]:
data = copy.deepcopy(dy_)
data = data[data['Hgb_F_hemoglobineid'].notna()]
data = data[data['Hgb_F_hemoglobineid']>0]

In [63]:
# hg_column = []
# for c in data.columns:
#     if 'hemoglobine' in c.lower():
#         hg_column.append(c)
        
# hg_column.remove('Hgb_F_hemoglobineid')
# data.drop(hg_column,inplace=True, axis=1)

In [64]:
hu_col = []
for c in data.columns:
    if 'HU' in c:
        hu_col.append(c)
data.drop(hu_col,inplace=True, axis=1)

In [65]:
df_list = []

g = data.groupby('SCCRIP_ID')
n = len(list(g.groups.keys()))
print(n)
for k in g.groups.keys():
    # print (k)
    tf = g.get_group(k)
    tf = tf.sort_values(by='date')
    # tf = tf.ffill()

    df_list.append(copy.deepcopy(tf))

1267


In [66]:
data = pd.concat(df_list)

data.to_pickle(result_dir + 'combined_raw_dy.pkl')

In [67]:
hu = read_sas('../data/Apply_MI_Zheng_2023_01/hu.sas7bdat')
hu.drop(['Age_initial_HU','Current_HU', 'Cohort_21Freeze'], inplace=True, axis=1)

In [68]:
data['date'] = pd.to_datetime(data['date'])
data.set_index('date',inplace=True)

In [69]:
########Create the HU treatment
data['HU_status'] = 0
data_copy = pd.DataFrame(columns = data.columns)
for i,r in hu.iterrows():
    ids = r['SCCRIP_ID']
    t_data = data[data['SCCRIP_ID'] == ids]
    t_data = t_data.sort_index()
    if r['Sum_HU'] == 1:
        if pd.isna(r['HU_END_DT1']):
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_Start_DT1']), 'HU_status'] = 1
        else:
            t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT1'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT1']))), 'HU_status'] = 1
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_END_DT1']), 'HU_status'] = 111
    if r['Sum_HU'] == 2:
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT1'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT1']))), 'HU_status'] = 1
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_END_DT1'])) & (t_data.index <= pd.to_datetime(r['HU_Start_DT2']))), 'HU_status'] = 111
        if pd.isna(r['HU_END_DT2']):
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_Start_DT2']), 'HU_status'] = 2
        else:
            t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT2'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT2']))), 'HU_status'] = 2
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_END_DT2']), 'HU_status'] = 222
    if r['Sum_HU'] == 3:
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT1'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT1']))), 'HU_status'] = 1
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_END_DT1'])) & (t_data.index <= pd.to_datetime(r['HU_Start_DT2']))), 'HU_status'] = 111
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT2'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT2']))), 'HU_status'] = 2
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_END_DT2'])) & (t_data.index <= pd.to_datetime(r['HU_Start_DT3']))), 'HU_status'] = 222
        if pd.isna(r['HU_END_DT3']):
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_Start_DT3']), 'HU_status'] = 3
        else:
            t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT3'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT3']))), 'HU_status'] = 3
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_END_DT3']), 'HU_status'] = 333
    if r['Sum_HU'] == 4:
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT1'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT1']))), 'HU_status'] = 1
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_END_DT1'])) & (t_data.index <= pd.to_datetime(r['HU_Start_DT2']))), 'HU_status'] = 111
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT2'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT2']))), 'HU_status'] = 2
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_END_DT2'])) & (t_data.index <= pd.to_datetime(r['HU_Start_DT3']))), 'HU_status'] = 222
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT3'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT3']))), 'HU_status'] = 3
        t_data.loc[((t_data.index >= pd.to_datetime(r['HU_END_DT3'])) & (t_data.index <= pd.to_datetime(r['HU_Start_DT4']))), 'HU_status'] = 333
        if pd.isna(r['HU_END_DT4']):
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_Start_DT4']), 'HU_status'] = 4
        else:
            t_data.loc[((t_data.index >= pd.to_datetime(r['HU_Start_DT4'])) & (t_data.index <= pd.to_datetime(r['HU_END_DT4']))), 'HU_status'] = 4
            t_data.loc[t_data.index >= pd.to_datetime(r['HU_END_DT4']), 'HU_status'] = 444
            
    data_copy = pd.concat([data_copy,t_data])
 

/lsf_tmp/272327336.tmpdir/ipykernel_693546/3725547470.py:45: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  data_copy = pd.concat([data_copy,t_data])
/lsf_tmp/272327336.tmpdir/ipykernel_693546/3725547470.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data_copy = pd.concat([data_copy,t_data])
/lsf_tmp/272327336.tmpdir/ipykernel_693546/3725547470.py:45: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns

In [70]:
data_copy = data_copy.reset_index().rename(columns={'index': 'date'})

In [71]:
data_copy2 = pd.DataFrame(columns = data_copy.columns)
g = data_copy.groupby('SCCRIP_ID')
for k in g.groups.keys():
    t = g.get_group(k)
    t = t.sort_values(by='date', ignore_index=True)
    change_from_zero = (t['HU_status'] == 1) & (t['HU_status'].shift(1) == 0)
    t.loc[change_from_zero, 'HU_status'] = 1110
    
    change_from_zero = (t['HU_status'] == 2) & ((t['HU_status'].shift(1) == 111) | (t['HU_status'].shift(1) == 1))
    t.loc[change_from_zero, 'HU_status'] = 2220
    
    change_from_zero = (t['HU_status'] == 3) & ((t['HU_status'].shift(1) == 222) | (t['HU_status'].shift(1) == 2))
    t.loc[change_from_zero, 'HU_status'] = 3330
    
    change_from_zero = (t['HU_status'] == 4) & ((t['HU_status'].shift(1) == 333) | (t['HU_status'].shift(1) == 3))
    t.loc[change_from_zero, 'HU_status'] = 4440
    
    data_copy2 = pd.concat([data_copy2, t], ignore_index=True)

/lsf_tmp/272327336.tmpdir/ipykernel_693546/1472931089.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data_copy2 = pd.concat([data_copy2, t], ignore_index=True)
/lsf_tmp/272327336.tmpdir/ipykernel_693546/1472931089.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  data_copy2 = pd.concat([data_copy2, t], ignore_index=True)
/lsf_tmp/272327336.tmpdir/ipykernel_693546/1472931089.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a fut

In [72]:
data_no_hu = data[~data['SCCRIP_ID'].isin(hu['SCCRIP_ID'].unique())]
data_no_hu = data_no_hu.reset_index().rename(columns={'Date': 'date'})
data_copy3 = pd.concat([data_copy2,data_no_hu])

In [73]:
# data_copy3['date'] = pd.to_datetime(data_copy3['date']).astype(int)

In [74]:
data_copy3.to_pickle(result_dir + 'LSTM_processed_data_dy.pkl')

In [75]:
del dy
del dy_
del data
del data_copy
del data_copy2

In [76]:
##########################################

In [77]:
ba = pd.read_pickle(ba_file)

sex_col = []
for c in ba.columns:
    if 'Sex' in c or 'sex' in c:
        sex_col.append(c)

In [78]:
hu_col = []
for c in ba.columns:
    if ('HU_' in c) or ('_hu' in c):
        hu_col.append(c)
ba.drop(hu_col, inplace=True, axis=1)

In [79]:
# d_col = []
# for col in ba.columns:
#     if ba[col].dtype == 'datetime64[ns]':
#         d_col.append(col)
#         ba[col] = pd.to_numeric(pd.to_datetime(ba[col]))

In [80]:
ped_col = []
for c in ba.columns:
    if 'sql' in c:
        # print(c)
        ped_col.append(c)
ba.drop(ped_col, inplace=True, axis=1)

In [81]:
mapping_dict_ba = encode_object_columns(ba)

with open(result_dir + 'mapping_dict_background.pkl', 'wb') as f:
    pickle.dump(mapping_dict_ba, f)

In [82]:
ba.to_pickle(result_dir + 'LSTM_processed_data_ba.pkl')

In [86]:
data_copy3.shape, ba.shape

((34927, 870), (1332, 45))